# BSI reducido en NHANES (v4.1): tiempo hasta el evento y pesos de la encuesta
Verificación adicional del cuaderno v3. NHANES 1999–2002, adultos ≥50 años con metilación; desenlace: **tiempo hasta la muerte** (meses desde el examen, seguimiento hasta 2019).

**Cómo usarlo:** *Entorno de ejecución → Ejecutar todo*. Todo se descarga solo del CDC. Si una celda se detiene con un aviso, mándame lo que imprime.

**Reglas fijadas ANTES de mirar resultados (v4)**
- Mismos modelos, mismo reloj (`epi_PhenoAge`) y mismas definiciones del BSI que en v3. No se cambió nada del índice.
- Modelos de Cox, validación cruzada 5×5 fuera de muestra, medida = **índice de concordancia (C)**.
- Primario: ΔC (M4 − M3) con IC95% por bootstrap pareado. **AGREGA** si el límite inferior > 0; **NO AGREGA** si el IC cae entero dentro de ±0.01; si no, **INCONCLUSO**.
- Se repite en casos completos (los 7 marcadores) y con el control de marcadores continuos (M4 vs M6), igual que en v3.
- Tiempo de seguimiento: meses desde el examen (si falta, desde la entrevista); las muertes en el mismo mes del examen se codifican con 0.5 mes y **no se excluyen** (corrección respecto a la primera versión del v4, que las había descartado).
- Sensibilidad con pesos: HR del BSI en Cox con el peso muestral del CDC para esta submuestra (`WTDN4YR`, según la documentación del CDC), con error estándar robusto.

Sigue siendo un BSI **reducido** y una sola foto transversal: **no prueba capacidad adaptativa (PE3)**.

In [1]:
!pip -q install lifelines

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.1/409.1 kB 4.2 MB/s eta 0:00:00


In [2]:
%%writefile bsi_nhanes.py
"""Valor anadido del BSI reducido sobre edad + reloj epigenetico + carga alostatica (NHANES 1999-2002, >=50 anos).
Todo lo que sigue esta FIJADO ANTES de mirar resultados (ver DECISION_RULE)."""
import re
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

SEED = 2026
N_BOOT = 2000
MARGIN = 0.01  # margen de equivalencia en AUC
DECISION_RULE = ("SUMA: limite inferior del IC95% de dAUC (M4-M3) > 0. "
                 "NO SUMA: IC95% de dAUC completamente dentro de (-0.01, +0.01). "
                 "Cualquier otro caso: INCONCLUSO.")

# Nombres candidatos de columnas (NHANES). Si alguno falta, check_columns() lo avisa y se detiene.
CANDIDATES = {
    "age": ["RIDAGEYR"], "sex": ["RIAGENDR"],
    "crp": ["LBXCRP", "LBDCRP"], "glucose": ["LBXGLU"], "insulin": ["LBXIN"],
    "tg": ["LBXTR"], "hdl": ["LBDHDL", "LBXHDD"], "hba1c": ["LBXGH"],
    "uma": ["URXUMA"], "ucr": ["URXUCR"],
}
CLOCK_REGEX = r"(pheno|grim|dunedin|hannum|horvath)"

def pick(df, names):
    for n in names:
        if n in df.columns:
            return n
    low = {c.lower(): c for c in df.columns}
    for n in names:
        if n.lower() in low:
            return low[n.lower()]
    return None

def check_columns(df):
    found, missing = {}, []
    for role, names in CANDIDATES.items():
        c = pick(df, names)
        (found.__setitem__(role, c) if c else missing.append(role))
    clocks = [c for c in df.columns if re.search(CLOCK_REGEX, c, re.I)]
    return found, missing, clocks

def zs(x):
    return (x - x.mean()) / x.std(ddof=0)

def build_scores(df, cols, clock_col):
    d = pd.DataFrame({"seqn": df["SEQN"]})
    d["age"] = df[cols["age"]]; d["female"] = (df[cols["sex"]] == 2).astype(int)
    crp, ins = np.log(df[cols["crp"]].clip(lower=1e-3)), np.log(df[cols["insulin"]].clip(lower=0.1))
    tg, glu = np.log(df[cols["tg"]].clip(lower=1)), df[cols["glucose"]]
    hba, hdl = df[cols["hba1c"]], df[cols["hdl"]]
    acr = np.log((df[cols["uma"]].clip(lower=0.1) / df[cols["ucr"]].clip(lower=1)) * 100)
    # reloj: aceleracion = residuo del reloj sobre la edad cronologica
    c = df[clock_col]; ok = c.notna() & d["age"].notna()
    slope, icpt = np.polyfit(d.loc[ok, "age"], c[ok], 1)
    accel = c - (icpt + slope * d["age"])
    # dominios (mayor = peor)
    d["dom_inflam"] = zs(pd.concat([zs(crp), zs(acr)], axis=1).mean(axis=1))
    d["dom_metab"] = zs(pd.concat([zs(glu), zs(ins), zs(tg), zs(hba), -zs(hdl)], axis=1).mean(axis=1))
    d["dom_epi"] = zs(accel)
    dom = d[["dom_inflam", "dom_metab", "dom_epi"]]
    d["bsi_mean"] = dom.mean(axis=1)    # primario: promedio con igual peso
    d["bsi_max"] = dom.max(axis=1)      # secundario: dominio mas alterado
    d["clock_accel"] = accel
    # carga alostatica simplificada: n de marcadores en el cuartil de peor riesgo
    risk = pd.concat([crp > crp.quantile(.75), acr > acr.quantile(.75), hba > hba.quantile(.75),
                      tg > tg.quantile(.75), hdl < hdl.quantile(.25),
                      glu > glu.quantile(.75), ins > ins.quantile(.75)], axis=1)
    d["allo_load"] = risk.sum(axis=1)
    return d

MODELS = {
    "M1 edad+sexo": ["age", "female"],
    "M2 +reloj": ["age", "female", "clock_accel"],
    "M3 +carga alostatica": ["age", "female", "clock_accel", "allo_load"],
    "M4 +BSI (promedio)": ["age", "female", "clock_accel", "allo_load", "bsi_mean"],
    "M5 +BSI (dominio max)": ["age", "female", "clock_accel", "allo_load", "bsi_max"],
}

def oof_predictions(d, y, folds=5, repeats=5):
    """Predicciones fuera de muestra, mismas particiones para todos los modelos."""
    out = {m: np.zeros((repeats, len(d))) for m in MODELS}
    for r in range(repeats):
        for tr, te in StratifiedKFold(folds, shuffle=True, random_state=SEED + r).split(d, y):
            for m, feats in MODELS.items():
                mod = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
                mod.fit(d.iloc[tr][feats], y[tr])
                out[m][r, te] = mod.predict_proba(d.iloc[te][feats])[:, 1]
    return {m: v.mean(axis=0) for m, v in out.items()}

def paired_boot_delta(y, p_new, p_old, n_boot=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed); n = len(y); ds = []
    for _ in range(n_boot):
        i = rng.integers(0, n, n)
        if y[i].min() == y[i].max():
            continue
        ds.append(roc_auc_score(y[i], p_new[i]) - roc_auc_score(y[i], p_old[i]))
    return np.percentile(ds, [2.5, 97.5])

def run(df_all, cols, clock_col, y_col="MORTSTAT"):
    d = build_scores(df_all, cols, clock_col)
    d["y"] = df_all[y_col].values
    d = d.dropna().reset_index(drop=True)
    y = d["y"].astype(int).values
    P = oof_predictions(d, y)
    rows = [{"modelo": m, "AUC (fuera de muestra)": round(roc_auc_score(y, p), 4)} for m, p in P.items()]
    res = pd.DataFrame(rows)
    out = {"n": len(d), "muertes": int(y.sum()), "tabla": res,
           "corr_dominios": d[["dom_inflam", "dom_metab", "dom_epi"]].corr().round(2),
           "corr_bsi_allo": round(float(d["bsi_mean"].corr(d["allo_load"])), 2)}
    for name, new in [("M4 +BSI (promedio)", "M4 +BSI (promedio)"), ("M5 +BSI (dominio max)", "M5 +BSI (dominio max)")]:
        lo, hi = paired_boot_delta(y, P[new], P["M3 +carga alostatica"])
        dauc = roc_auc_score(y, P[new]) - roc_auc_score(y, P["M3 +carga alostatica"])
        verdict = ("SUMA" if lo > 0 else "NO SUMA" if (lo > -MARGIN and hi < MARGIN) else "INCONCLUSO")
        out[name] = {"dAUC vs M3": round(dauc, 4), "IC95%": (round(lo, 4), round(hi, 4)), "veredicto": verdict}
    return out

Writing bsi_nhanes.py


In [3]:
import os, glob, requests, numpy as np, pandas as pd
import bsi_nhanes as B
os.makedirs('data', exist_ok=True)
for f in glob.glob('data/*'): os.remove(f)

BASE_URL = 'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/{y}/DataFiles/{n}.xpt'
NAMES = {
 '1999': ['DEMO','LAB11','LAB10AM','LAB13AM','LAB13','LAB10','LAB16'],
 '2001': ['DEMO_B','L11_B','L10AM_B','L13AM_B','L13_B','L10_B','L16_B'],
}
MORT = 'https://ftp.cdc.gov/pub/Health_Statistics/NCHS/datalinkage/linked_mortality/'
MORTFILES = ['NHANES_1999_2000_MORT_2019_PUBLIC.dat','NHANES_2001_2002_MORT_2019_PUBLIC.dat']
DNAM_URL = 'https://wwwn.cdc.gov/nchs/data/nhanes/dnam/dnmepi.sas7bdat'

def es_xport(b): return b[:20].startswith(b'HEADER RECORD')
fallan = []
def bajar(url, dest, valido):
    r = requests.get(url, timeout=120)
    if r.status_code == 200 and valido(r.content):
        open(dest, 'wb').write(r.content); print('OK   ', os.path.basename(dest), len(r.content), 'bytes')
    else:
        fallan.append(os.path.basename(dest)); print('FALLA', os.path.basename(dest), r.status_code, r.content[:40])

for y, ns in NAMES.items():
    for n in ns: bajar(BASE_URL.format(y=y, n=n), f'data/{n}.XPT', es_xport)
for n in MORTFILES:
    bajar(MORT + n, f'data/{n}', lambda b: len(b) > 100000 and not b[:15].lower().startswith(b'<!doc'))
bajar(DNAM_URL, 'data/dnmepi.sas7bdat', lambda b: len(b) > 100000 and not b[:15].lower().startswith(b'<!doc'))
print('\nFallan:', fallan)
assert not fallan, 'Alguna descarga fallo; mandame esta lista.'

OK    DEMO.XPT 11500560 bytes
OK    LAB11.XPT 469040 bytes
OK    LAB10AM.XPT 210960 bytes
OK    LAB13AM.XPT 221040 bytes
OK    LAB13.XPT 335200 bytes
OK    LAB10.XPT 109200 bytes
OK    LAB16.XPT 314240 bytes
OK    DEMO_B.XPT 3273520 bytes
OK    L11_B.XPT 446240 bytes
OK    L10AM_B.XPT 236480 bytes
OK    L13AM_B.XPT 251760 bytes
OK    L13_B.XPT 371920 bytes
OK    L10_B.XPT 120160 bytes
OK    L16_B.XPT 347360 bytes
OK    NHANES_1999_2000_MORT_2019_PUBLIC.dat 487666 bytes
OK    NHANES_2001_2002_MORT_2019_PUBLIC.dat 540362 bytes
OK    dnmepi.sas7bdat 1245184 bytes

Fallan: []


In [4]:
def read_cycle(names):
    base = None
    for n in names:
        t = pd.read_sas(f'data/{n}.XPT', format='xport', encoding='utf-8')
        base = t if base is None else base.merge(t, on='SEQN', how='outer', suffixes=('','_dup'))
    return base
lab = pd.concat([read_cycle(NAMES['1999']), read_cycle(NAMES['2001'])], ignore_index=True)
epi = pd.read_sas('data/dnmepi.sas7bdat', encoding='utf-8')
epi.columns = [c if c == 'SEQN' else 'epi_' + c for c in epi.columns]
cols, miss, clocks = B.check_columns(lab.merge(epi, on='SEQN'))
print('roles que faltan:', miss); print('relojes disponibles:', clocks)
CLOCK = 'epi_PhenoAge'   # fijado antes de ver resultados; no lo cambies
assert not miss and CLOCK in epi.columns, 'Falta una variable o el reloj; mandame lo impreso.'
print('Reloj elegido:', CLOCK)
print('Peso WTDN4YR disponible:', 'epi_WTDN4YR' in epi.columns)

roles que faltan: []
relojes disponibles: ['epi_HorvathAge', 'epi_HannumAge', 'epi_PhenoAge', 'epi_GrimAgeMort', 'epi_GrimAge2Mort', 'epi_HorvathTelo', 'epi_DunedinPoAm']
Reloj elegido: epi_PhenoAge
Peso WTDN4YR disponible: True


In [5]:
spec = [(0,14),(14,15),(15,16),(16,19),(43,46),(46,49)]
names = ['SEQN','ELIGSTAT','MORTSTAT','UCOD','PERMTH_INT','PERMTH_EXM']
m = pd.concat([pd.read_fwf(f'data/{f}', colspecs=spec, names=names) for f in MORTFILES], ignore_index=True)
for c in ['SEQN','ELIGSTAT','MORTSTAT','PERMTH_EXM','PERMTH_INT']:
    m[c] = pd.to_numeric(m[c], errors='coerce')
m = m[m['ELIGSTAT'] == 1]
print('Mortalidad:', m['MORTSTAT'].value_counts(dropna=False).to_dict())
assert set(m['MORTSTAT'].dropna().unique()) <= {0,1}, 'El ancho fijo no calza; mandame esto.'
print('Meses de seguimiento (PERMTH_EXM):'); print(m['PERMTH_EXM'].describe().round(1).to_string())
assert m['PERMTH_EXM'].min() >= 0 and m['PERMTH_EXM'].max() <= 300, 'Los meses no calzan; mandame esto.'
df = lab.merge(epi, on='SEQN').merge(m[['SEQN','MORTSTAT','PERMTH_EXM','PERMTH_INT']], on='SEQN')
df = df[df[cols['age']] >= 50].copy()
# Tiempo de seguimiento: meses desde el examen. Si falta, meses desde la entrevista.
# Tiempo 0 (muerte en el mismo mes del examen) NO se descarta: se codifica 0.5 mes.
print('Tiempo vacio:', int(df['PERMTH_EXM'].isna().sum()), '| tiempo = 0:', int((df['PERMTH_EXM'] == 0).sum()))
df['PERMTH_EXM'] = df['PERMTH_EXM'].fillna(df['PERMTH_INT'])
df.loc[df['PERMTH_EXM'] == 0, 'PERMTH_EXM'] = 0.5
print('Tiempo vacio despues de corregir:', int(df['PERMTH_EXM'].isna().sum()))
print('Participantes >=50 con metilacion y mortalidad:', len(df), '| muertes:', int(df['MORTSTAT'].sum()))

Mortalidad: {0.0: 8133, 1.0: 3299}
Meses de seguimiento (PERMTH_EXM):
count    10491.0
mean        26.1
std         18.7
min          0.0
25%         11.0
50%         24.0
75%         38.0
max         99.0
Tiempo vacio: 60 | tiempo = 0: 123
Tiempo vacio despues de corregir: 6
Participantes >=50 con metilacion y mortalidad: 4446 | muertes: 2493


## Funciones de Cox

In [6]:
import numpy as np, pandas as pd, warnings
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
from sklearn.model_selection import StratifiedKFold
warnings.filterwarnings("ignore")

def cox_oof(d, feats_by_model, folds=5, repeats=5):
    """Riesgo (log partial hazard) fuera de muestra, mismas particiones para todos los modelos."""
    y = d["event"].values
    out = {m: np.zeros((repeats, len(d))) for m in feats_by_model}
    for r in range(repeats):
        for tr, te in StratifiedKFold(folds, shuffle=True, random_state=B.SEED + r).split(d, y):
            for m, f in feats_by_model.items():
                cph = CoxPHFitter(penalizer=1e-4)
                cph.fit(d.iloc[tr][f + ["time", "event"]], duration_col="time", event_col="event")
                out[m][r, te] = cph.predict_log_partial_hazard(d.iloc[te][f]).values
    return {m: v.mean(axis=0) for m, v in out.items()}

def cidx(t, e, risk):
    return concordance_index(t, -risk, e)

def boot_dC(t, e, r_new, r_old, n_boot=2000, seed=B.SEED):
    rng = np.random.default_rng(seed); n = len(t); ds = []
    for _ in range(n_boot):
        i = rng.integers(0, n, n)
        if e[i].sum() < 5: continue
        ds.append(cidx(t[i], e[i], r_new[i]) - cidx(t[i], e[i], r_old[i]))
    return np.percentile(ds, [2.5, 97.5])

def veredicto_C(lo, hi, margin=0.01):
    return "AGREGA" if lo > 0 else "NO AGREGA" if (lo > -margin and hi < margin) else "INCONCLUSO"

def comparar(t, e, R, pares):
    for nuevo, viejo, titulo in pares:
        lo, hi = boot_dC(t, e, R[nuevo], R[viejo])
        dC = cidx(t, e, R[nuevo]) - cidx(t, e, R[viejo])
        print(f"{titulo}: dC = {dC:.4f}  IC95% = ({lo:.4f}, {hi:.4f})  -> {veredicto_C(lo, hi)}")

## Análisis primario (misma población que v3) y HR con pesos

In [7]:
d = B.build_scores(df, cols, CLOCK)
d["event"] = df["MORTSTAT"].values
d["time"] = df["PERMTH_EXM"].values
d["w"] = df["epi_WTDN4YR"].values if "epi_WTDN4YR" in df.columns else np.nan
d_all = d.copy()
d = d.dropna(subset=[c for c in d.columns if c != "w"]).reset_index(drop=True)
assert (d["time"] > 0).all()
t, e = d["time"].values, d["event"].astype(int).values
print(f"n analizado = {len(d)}, muertes = {int(e.sum())}  (en v3 fueron 2532 y 1361)")

BASE = ["age", "female", "clock_accel"]
FE = {
    "M1 edad+sexo": ["age", "female"],
    "M2 +reloj": BASE,
    "M3 +carga alostatica": BASE + ["allo_load"],
    "M4 +BSI (promedio)": BASE + ["allo_load", "bsi_mean"],
    "M5 +BSI (dominio max)": BASE + ["allo_load", "bsi_max"],
}
R = cox_oof(d, FE)
tabla = pd.DataFrame([{"modelo": k, "C (fuera de muestra)": round(cidx(t, e, v), 4)} for k, v in R.items()])
display(tabla); tabla.to_csv("tabla_C.csv", index=False)
print("PRIMARIO (regla fijada):")
comparar(t, e, R, [("M4 +BSI (promedio)", "M3 +carga alostatica", "M4 - M3 (BSI promedio sobre carga alostatica)"),
                   ("M5 +BSI (dominio max)", "M3 +carga alostatica", "M5 - M3 (secundario: dominio mas alterado)")])

# HR del BSI por 1 DE, ajustado (ajuste en toda la muestra, no fuera de muestra)
dd = d.copy(); dd["bsi_mean_z"] = (dd["bsi_mean"] - dd["bsi_mean"].mean()) / dd["bsi_mean"].std()
f4 = BASE + ["allo_load", "bsi_mean_z"]
cph = CoxPHFitter(penalizer=1e-4).fit(dd[f4 + ["time", "event"]], "time", "event")
s = cph.summary.loc["bsi_mean_z"]
print(f"\nHR del BSI (por 1 DE), ajustado por edad, sexo, reloj y carga alostatica: {s['exp(coef)']:.3f}  IC95% ({s['exp(coef) lower 95%']:.3f}, {s['exp(coef) upper 95%']:.3f})")
try:
    from lifelines.statistics import proportional_hazard_test
    pht = proportional_hazard_test(cph, dd[f4 + ["time", "event"]], time_transform="rank")
    print("Prueba de riesgos proporcionales (p; p<0.05 sugiere que el HR cambia con el tiempo):")
    print(pht.summary[["p"]].round(4).to_string())
except Exception as ex:
    print("No se pudo correr la prueba de riesgos proporcionales:", ex)

# Sensibilidad con pesos (WTDN4YR) y error robusto
if d["w"].notna().any():
    dw = dd[dd["w"] > 0].copy()
    cphw = CoxPHFitter(penalizer=1e-4).fit(dw[f4 + ["time", "event", "w"]], "time", "event", weights_col="w", robust=True)
    sw = cphw.summary.loc["bsi_mean_z"]
    print(f"HR del BSI con pesos WTDN4YR (n con peso>0 = {len(dw)}): {sw['exp(coef)']:.3f}  IC95% ({sw['exp(coef) lower 95%']:.3f}, {sw['exp(coef) upper 95%']:.3f})")
    print("Nota: WTDN4YR corresponde a la submuestra con metilacion. Quienes ademas tienen los marcadores de ayuno son un subconjunto, asi que este peso es aproximado para ellos.")
else:
    print("No hay columna de pesos WTDN4YR; se omite la sensibilidad con pesos.")

n analizado = 2530, muertes = 1359  (en v3 fueron 2532 y 1361)


,modelo,C (fuera de muestra)
0,M1 edad+sexo,0.7078
1,M2 +reloj,0.7124
2,M3 +carga alostatica,0.7161
3,M4 +BSI (promedio),0.7232
4,M5 +BSI (dominio max),0.7258


PRIMARIO (regla fijada):
M4 - M3 (BSI promedio sobre carga alostatica): dC = 0.0071  IC95% = (0.0033, 0.0108)  -> AGREGA
M5 - M3 (secundario: dominio mas alterado): dC = 0.0097  IC95% = (0.0056, 0.0141)  -> AGREGA

HR del BSI (por 1 DE), ajustado por edad, sexo, reloj y carga alostatica: 1.332  IC95% (1.207, 1.471)
Prueba de riesgos proporcionales (p; p<0.05 sugiere que el HR cambia con el tiempo):
                  p
age          0.0000
allo_load    0.1466
bsi_mean_z   0.0018
clock_accel  0.7438
female       0.8123
HR del BSI con pesos WTDN4YR (n con peso>0 = 2530): 1.299  IC95% (1.091, 1.547)
Nota: WTDN4YR corresponde a la submuestra con metilacion. Quienes ademas tienen los marcadores de ayuno son un subconjunto, asi que este peso es aproximado para ellos.


## Casos completos y control con marcadores continuos

In [8]:
dc = d_all.copy()
dc["l_crp"] = np.log(df[cols["crp"]].clip(lower=1e-3)).values
dc["l_acr"] = np.log((df[cols["uma"]].clip(lower=0.1) / df[cols["ucr"]].clip(lower=1)) * 100).values
dc["hba1c"] = df[cols["hba1c"]].values
dc["l_tg"] = np.log(df[cols["tg"]].clip(lower=1)).values
dc["hdl"] = df[cols["hdl"]].values
dc["glu"] = df[cols["glucose"]].values
dc["l_ins"] = np.log(df[cols["insulin"]].clip(lower=0.1)).values
dc = dc.drop(columns=["w"]).dropna().reset_index(drop=True)
assert (dc["time"] > 0).all()
tc, ec = dc["time"].values, dc["event"].astype(int).values
print(f"n (casos completos) = {len(dc)}, muertes = {int(ec.sum())}  (en v3 fueron 1228 y 644)")
MK = ["l_crp", "l_acr", "hba1c", "l_tg", "hdl", "glu", "l_ins"]
FE2 = {
    "M3 +carga alostatica": BASE + ["allo_load"],
    "M4 +BSI (promedio)": BASE + ["allo_load", "bsi_mean"],
    "M6 +marcadores continuos": BASE + ["allo_load"] + MK,
    "M7 +3 dominios por separado": BASE + ["allo_load", "dom_inflam", "dom_metab", "dom_epi"],
}
R2 = cox_oof(dc, FE2)
display(pd.DataFrame([{"modelo": k, "C (fuera de muestra)": round(cidx(tc, ec, v), 4)} for k, v in R2.items()]))
comparar(tc, ec, R2, [
    ("M4 +BSI (promedio)", "M3 +carga alostatica", "Sensibilidad: M4 - M3 (casos completos)"),
    ("M4 +BSI (promedio)", "M6 +marcadores continuos", "CONTROL  BSI sobre marcadores continuos (M4 - M6)"),
    ("M6 +marcadores continuos", "M3 +carga alostatica", "M6 sobre carga alostatica (M6 - M3)"),
    ("M7 +3 dominios por separado", "M4 +BSI (promedio)", "3 dominios por separado vs BSI promedio (M7 - M4)")])

n (casos completos) = 1228, muertes = 644  (en v3 fueron 1228 y 644)


,modelo,C (fuera de muestra)
0,M3 +carga alostatica,0.7195
1,M4 +BSI (promedio),0.7249
2,M6 +marcadores continuos,0.7273
3,M7 +3 dominios por separado,0.7287


Sensibilidad: M4 - M3 (casos completos): dC = 0.0054  IC95% = (0.0015, 0.0095)  -> AGREGA
CONTROL  BSI sobre marcadores continuos (M4 - M6): dC = -0.0025  IC95% = (-0.0088, 0.0039)  -> NO AGREGA
M6 sobre carga alostatica (M6 - M3): dC = 0.0079  IC95% = (0.0009, 0.0154)  -> AGREGA
3 dominios por separado vs BSI promedio (M7 - M4): dC = 0.0039  IC95% = (-0.0012, 0.0092)  -> NO AGREGA


## Cómo leer esto
- Compara con v3: si el **tamaño** de la mejora (ΔC ≈ ΔAUC ≈ 0.004) y el patrón del control (M4 vs M6) se mantienen con tiempo hasta el evento, la conclusión de v3 no depende de haber usado "muerte sí/no".
- El **HR con pesos** muestra si el BSI sigue asociado a la mortalidad al tener en cuenta el diseño de la encuesta (aproximado, ver nota).
- **Límites que siguen**: BSI reducido (sin cortisol ni BDNF), PhenoAge comparte marcadores con el BSI, el IC bootstrap no incluye la variabilidad del ajuste del modelo, los pesos no corrigen el subconjunto con marcadores de ayuno, y una foto transversal **no prueba capacidad adaptativa (PE3)**.